# 04 · ColBERT — late interaction (token-level matching)
A middle ground between bi-encoders (one vector per doc, fast but coarse) and cross-encoders (joint, accurate but slow). ColBERT keeps **one vector per token** and matches at the token level — the 'late interaction' idea.

## 1. One vector per token, not per document
```
  bi-encoder:   whole doc  -> ONE vector           (loses word-level detail)
  ColBERT:      each token -> its OWN vector        (keeps word-level detail)

  scoring (MaxSim): for each QUERY token, find its best-matching DOC token,
                    then SUM those best matches.
```

## 2. MaxSim, by hand
```
  query tokens:  [waterproof] [jacket]
  doc tokens:    [this][jacket][is][waterproof][and][windproof]

  waterproof -> best match = doc 'waterproof'  (sim 1.0)
  jacket     -> best match = doc 'jacket'      (sim 1.0)
  MaxSim score = 1.0 + 1.0 = 2.0
```
Because it matches token-by-token, ColBERT catches when a doc contains the exact concepts a query asks about, even if the overall 'average' vector would look diluted.

In [ ]:
import numpy as np, re
def toks(s): return re.findall(r"[a-z]+", s.lower())

# toy token embeddings: each distinct word -> a random unit vector (deterministic)
rng = np.random.default_rng(42)
def tok_vecs(words, table):
    out=[]
    for w in words:
        if w not in table:
            v = rng.standard_normal(16); table[w]=v/np.linalg.norm(v)
        out.append(table[w])
    return np.array(out)

table={}
def maxsim(query, doc):
    q = tok_vecs(toks(query), table)
    d = tok_vecs(toks(doc), table)
    sims = q @ d.T                      # every query token vs every doc token
    return float(sims.max(axis=1).sum())  # best doc match per query token, summed

query="waterproof jacket"
docs=["this jacket is waterproof and windproof",
      "our running shoes are lightweight",
      "a waterproof phone case"]
for d in sorted(docs, key=lambda d:-maxsim(query,d)):
    print(f"  {maxsim(query,d):.2f}  {d}")

**Observe:** the jacket doc wins because BOTH query tokens ('waterproof','jacket') find strong matches; the phone case matches only 'waterproof'; the shoes match neither. Token-level matching captures this where a single averaged vector might not.

**Where ColBERT sits:**
```
  bi-encoder      ColBERT (late interaction)     cross-encoder
  fast, coarse -> more accurate, still scalable -> most accurate, slowest
```
The cost is storage — one vector per token is many more vectors. Your course's `l-rag-colbert` lab and the LangChain book's ColBERT file use this; it's a strong retriever when you can afford the index size.